# Natural Language Inference

In [13]:
import torch
import warnings
import numpy as np
import pandas as pd
from transformers import T5ForConditionalGeneration, T5Tokenizer, T5Config
from sentence_transformers import SentenceTransformer, util
from rank_bm25 import BM25Okapi
from sklearn.metrics import accuracy_score

# ==========================================
# 0. DATA LOADING
# ==========================================

# Suppress potential warnings from the LLM or other libraries
warnings.filterwarnings("ignore")

# --- 1. Load the Full Dataset ---
print("Loading the full train dataset...")
data = []
file_path = 'C:\\Users\\brand\\Downloads\\NTU FYP Dataset\\Natural Language Inference\\anli_v1.0\\R1\\train.jsonl'
with open(file_path, 'r', encoding='utf-8') as f:
    for line in f:
        data.append(json.loads(line))

df = pd.DataFrame(data)
print(f"Dataset loaded. Total records: {len(df)}")


print("Loading the full test dataset...")
data = []
file_path = 'C:\\Users\\brand\\Downloads\\NTU FYP Dataset\\Natural Language Inference\\anli_v1.0\\R1\\test.jsonl'
with open(file_path, 'r', encoding='utf-8') as f:
    for line in f:
        data.append(json.loads(line))

df_1 = pd.DataFrame(data)
print(f"Dataset loaded. Total records: {len(df_1)}")

# ==========================================
# 1. SETUP: LOAD MODELS
# ==========================================
# A. Load Your Local Fine-Tuned T5 Model
model_path = "Downloads/final_student_model"  # Point to your local folder
print(f"Loading Fine-Tuned T5 Model from: {model_path}...")

try:
    # Force 'Base' config to fix shape mismatch (Your fix)
    my_config = T5Config.from_pretrained("google/flan-t5-base")
    t5_model = T5ForConditionalGeneration.from_pretrained(model_path, config=my_config)
    
    # Use original Google tokenizer (Your fix)
    tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-base")
    
    # Optimization: Move to GPU if available
    device = "cuda" if torch.cuda.is_available() else "cpu"
    t5_model.to(device)
    print(f"✅ T5 Model loaded successfully on {device}!")
except Exception as e:
    print(f"❌ Error loading T5 Model: {e}")
    print("Ensure 'pytorch_model.bin' exists in the specified folder.")

# B. Load Sentence Transformer (Required for Cosine & MMR Selection)
print("Loading SentenceTransformer for Exemplar Retrieval...")
retrieval_model = SentenceTransformer('all-MiniLM-L6-v2') 

# ==========================================
# 2. DATA PREPARATION & INDEXING
# ==========================================
# We need to index the training data ONCE to use it for retrieval
print("\n--- Indexing Training Data (Knowledge Base) ---")
train_texts = [f"{row['context']} {row['hypothesis']}" for _, row in df.iterrows()]
train_data_store = df.to_dict('records') 

# 1. Compute Embeddings for Cosine/MMR
print("Computing Embeddings (for Cosine & MMR)...")
train_embeddings = retrieval_model.encode(train_texts, convert_to_tensor=True, show_progress_bar=True)

# 2. Tokenize for BM25 (Complex Selection)
print("Tokenizing for BM25 (Complex Selection)...")
tokenized_corpus = [doc.split(" ") for doc in train_texts]
bm25 = BM25Okapi(tokenized_corpus)

# Sample Test Data
test_df_sample = df_1.sample(n=150, random_state=42).copy()
print(f"Evaluation set ready: {len(test_df_sample)} test cases.")

# ==========================================
# 3. HELPER FUNCTIONS (SELECTION LOGIC)
# ==========================================
def mmr_logic(query_embedding, candidate_embeddings, top_k=5, diversity=0.5):
    """Core MMR Logic: Balances Similarity and Diversity"""
    # 1. Calculate similarity between query and all candidates
    candidate_similarity = util.cos_sim(query_embedding, candidate_embeddings)[0]
    candidate_indices = list(range(len(candidate_embeddings)))
    selected_indices = [int(torch.argmax(candidate_similarity))] # Start with most similar
    candidate_indices.remove(selected_indices[0])
    
    for _ in range(top_k - 1):
        if not candidate_indices: break
        # Extract similarities
        cand_to_query_sim = candidate_similarity[candidate_indices]
        cand_to_selected_sim = util.cos_sim(candidate_embeddings[candidate_indices], candidate_embeddings[selected_indices])
        cand_to_selected_max_sim, _ = torch.max(cand_to_selected_sim, dim=1)
        
        # MMR Formula
        mmr_score = (1 - diversity) * cand_to_query_sim - diversity * cand_to_selected_max_sim
        best_candidate_idx = candidate_indices[torch.argmax(mmr_score).item()]
        
        selected_indices.append(best_candidate_idx)
        candidate_indices.remove(best_candidate_idx)
    return selected_indices

def retrieve_exemplars(method, query_text, k=5):
    """Switch function to handle the 3 selection methods"""
    selected_indices = []
    
    if method == 'Cosine Similarity':
        query_embedding = retrieval_model.encode(query_text, convert_to_tensor=True)
        # Standard Semantic Search (Pure Cosine)
        hits = util.semantic_search(query_embedding, train_embeddings, top_k=k)[0]
        selected_indices = [hit['corpus_id'] for hit in hits]

    elif method == 'MMR Selection':
        query_embedding = retrieval_model.encode(query_text, convert_to_tensor=True)
        # 1. Fetch larger pool (Top-20)
        hits = util.semantic_search(query_embedding, train_embeddings, top_k=20)[0]
        pool_indices = [hit['corpus_id'] for hit in hits]
        pool_embeddings = train_embeddings[pool_indices]
        # 2. Apply MMR Re-ranking
        mmr_relative_indices = mmr_logic(query_embedding, pool_embeddings, top_k=k, diversity=0.5)
        selected_indices = [pool_indices[i] for i in mmr_relative_indices]

    elif method == 'Complex (BM25)':
        tokenized_query = query_text.split(" ")
        scores = bm25.get_scores(tokenized_query)
        selected_indices = np.argsort(scores)[::-1][:k]

    return [train_data_store[i] for i in selected_indices]

# ==========================================
# 4. MAIN EVALUATION LOOP
# ==========================================
methods_to_test = ['Cosine Similarity', 'MMR Selection', 'Complex (BM25)']
results_summary = {}

for method in methods_to_test:
    print(f"\n🚀 Running Method: {method}...")
    predictions = []
    
    # We use enumerate with a counter to track index cleanly
    for counter, (index, row) in enumerate(test_df_sample.iterrows()):
        # A. Define Query
        current_pair_text = f"{row['context']} {row['hypothesis']}"
        
        # B. Retrieve 5-Shot Examples
        exemplars = retrieve_exemplars(method, current_pair_text, k=5)
        
        # C. Construct Prompt
        # We assume your T5 model benefits from seeing the examples in the input
        input_text = ""
        for ex in exemplars:
            input_text += f"nli premise: {ex['context']} hypothesis: {ex['hypothesis']} label: {ex['label']} "
        
        # Add the actual target at the end
        input_text += f"nli premise: {row['context']} hypothesis: {row['hypothesis']}"
        
        # D. Run Inference
        try:
            inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=1024).to(device)
            with torch.no_grad():
                outputs = t5_model.generate(**inputs, max_new_tokens=5)
            
            result_text = tokenizer.decode(outputs[0], skip_special_tokens=True).strip().lower()
            
            # Map Output to Label
            pred_char = 'error'
            if 'entailment' in result_text or result_text.startswith('e'): pred_char = 'e'
            elif 'contradiction' in result_text or result_text.startswith('c'): pred_char = 'c'
            elif 'neutral' in result_text or result_text.startswith('n'): pred_char = 'n'
            
            predictions.append(pred_char)
            
            # --- UPDATED: PRINT EVERY 10 ROWS ---
            if (counter + 1) % 10 == 0:
                print(f"   [{method}] Row {counter+1}/{len(test_df_sample)} | True: {row['label']} -> Pred: {pred_char} (Raw: {result_text})")
            
        except Exception as e:
            predictions.append('error')
            print(f"   Error on row {counter}: {e}")

    # E. Calculate Accuracy for this Method
    test_df_sample[f'pred_{method}'] = predictions
    valid_results = test_df_sample[test_df_sample[f'pred_{method}'].isin(['e', 'n', 'c'])]
    
    if not valid_results.empty:
        acc = accuracy_score(valid_results['label'], valid_results[f'pred_{method}'])
        results_summary[method] = acc
        print(f"   👉 Final Accuracy for {method}: {acc:.2%}")
    else:
        print(f"   ⚠️ No valid predictions for {method}")

# ==========================================
# 5. FINAL REPORT
# ==========================================
print("\n" + "="*40)
print("FINAL BENCHMARK RESULTS")
print("="*40)
for method, acc in results_summary.items():
    print(f"{method}: {acc:.2%}")

# Save detailed results
csv_filename = 'T5_benchmark_NLI_results.csv'
test_df_sample.to_csv(csv_filename, index=False)
print(f"\nDetailed CSV saved to '{csv_filename}'")

Loading the full train dataset...
Dataset loaded. Total records: 16946
Loading the full test dataset...
Dataset loaded. Total records: 1000
Loading Fine-Tuned T5 Model from: Downloads/final_student_model...
✅ T5 Model loaded successfully on cpu!
Loading SentenceTransformer for Exemplar Retrieval...

--- Indexing Training Data (Knowledge Base) ---
Computing Embeddings (for Cosine & MMR)...


Batches:   0%|          | 0/530 [00:00<?, ?it/s]

Tokenizing for BM25 (Complex Selection)...
Evaluation set ready: 150 test cases.

🚀 Running Method: Cosine Similarity...
   [Cosine Similarity] Row 10/150 | True: c -> Pred: e (Raw: entailment)
   [Cosine Similarity] Row 20/150 | True: c -> Pred: e (Raw: entailment)
   [Cosine Similarity] Row 30/150 | True: c -> Pred: e (Raw: entailment)
   [Cosine Similarity] Row 40/150 | True: e -> Pred: e (Raw: entailment)
   [Cosine Similarity] Row 50/150 | True: e -> Pred: e (Raw: entailment)
   [Cosine Similarity] Row 60/150 | True: c -> Pred: e (Raw: entailment)
   [Cosine Similarity] Row 70/150 | True: e -> Pred: e (Raw: entailment)
   [Cosine Similarity] Row 80/150 | True: c -> Pred: e (Raw: entailment)
   [Cosine Similarity] Row 90/150 | True: c -> Pred: e (Raw: entailment)
   [Cosine Similarity] Row 100/150 | True: n -> Pred: c (Raw: contradiction)
   [Cosine Similarity] Row 110/150 | True: e -> Pred: e (Raw: entailment)
   [Cosine Similarity] Row 120/150 | True: n -> Pred: c (Raw: contradic

# Commonsense Reasoning

In [14]:
import pandas as pd
import torch
import numpy as np
import warnings
from sklearn.metrics import accuracy_score
from sklearn.metrics.pairwise import euclidean_distances
from sklearn.metrics.pairwise import cosine_similarity
import time
from transformers import T5ForConditionalGeneration, T5Tokenizer, T5Config

# Suppress warnings
warnings.filterwarnings("ignore")

# DEVICE CONFIGURATION
# Use GPU if available for faster T5 generation
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running on device: {device}")


print("Loading CommonsenseQA dataset from Hugging Face...")

splits = {
    'train': 'data/train-00000-of-00001.parquet', 
    'validation': 'data/validation-00000-of-00001.parquet'
}

# We use validation as the 'test' set
df_val = pd.read_parquet("hf://datasets/tau/commonsense_qa/" + splits["validation"])
print(f"Validation dataset loaded. Total records: {len(df_val)}")

def format_choices(choices_dict):
    """Formats choices for the input prompt (e.g., (A) apple (B) banana)"""
    labels = list(choices_dict['label'])
    texts = list(choices_dict['text'])
    formatted = [f"({l}) {t}" for l, t in zip(labels, texts)]
    return " ".join(formatted)

def get_correct_text(row):
    """Gets the text of the correct answer for embedding comparison"""
    labels = list(row['choices']['label'])
    idx = labels.index(row['answerKey'])
    return row['choices']['text'][idx]

print("Preprocessing columns...")
df_val['formatted_choices'] = df_val['choices'].apply(format_choices)
df_val['correct_answer_text'] = df_val.apply(get_correct_text, axis=1)
df = df_val.copy()


# ==========================================
# 2. LOAD YOUR CUSTOM T5 MODEL
# ==========================================
model_path = "Downloads/final_student_model"  # Point to your local folder

print(f"\nLoading your fine-tuned model from: {model_path}")

# Force 'Base' config to fix shape mismatch (as per your fix)
my_config = T5Config.from_pretrained("google/flan-t5-base")
model = T5ForConditionalGeneration.from_pretrained(model_path, config=my_config)
model.to(device) # Move model to GPU if available

print("Loading tokenizer (google/flan-t5-base)...")
tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-base")


# ==========================================
# 3. GENERATE EMBEDDINGS (Using T5 Encoder)
# ==========================================
def get_t5_embeddings(texts, model, tokenizer, batch_size=16):
    """
    Generates embeddings by mean-pooling the last hidden state of the T5 Encoder.
    """
    model.eval()
    all_embeddings = []
    print(f"Generating embeddings for {len(texts)} texts using T5 Encoder...")
    
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        
        # Tokenize
        inputs = tokenizer(batch_texts, return_tensors="pt", padding=True, truncation=True, max_length=128)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        with torch.no_grad():
            # Pass through the ENCODER only
            encoder_outputs = model.encoder(**inputs)
            last_hidden_state = encoder_outputs.last_hidden_state
            
            # Mean Pooling
            input_mask_expanded = inputs['attention_mask'].unsqueeze(-1).expand(last_hidden_state.size()).float()
            sum_embeddings = torch.sum(last_hidden_state * input_mask_expanded, 1)
            sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
            
            mean_embeddings = sum_embeddings / sum_mask
            all_embeddings.append(mean_embeddings.cpu())
            
    return torch.cat(all_embeddings, dim=0)

# Generate Embeddings
q_embeddings = get_t5_embeddings(df['question'].tolist(), model, tokenizer)
a_embeddings = get_t5_embeddings(df['correct_answer_text'].tolist(), model, tokenizer)

# Convert to numpy for metric calculations
q_np = q_embeddings.numpy()
a_np = a_embeddings.numpy()

print("\nComputing retrieval metrics...")

# --- Metric 1: Cosine Similarity ---
cosine_vals = []
for i in range(len(q_np)):
    sim = cosine_similarity(q_np[i].reshape(1, -1), a_np[i].reshape(1, -1))[0][0]
    cosine_vals.append(sim)
df['cosine_similarity'] = cosine_vals

# --- Metric 2: Euclidean Distance (Your "Complex" Criterion) ---
dists = []
for i in range(len(q_np)):
    d = euclidean_distances(q_np[i].reshape(1, -1), a_np[i].reshape(1, -1))[0][0]
    dists.append(d)
df['euclidean_distance'] = dists

# --- Metric 3: MMR Selection Logic ---
def mmr_selection(df, doc_embeddings, relevance_scores, top_n=150, lambda_param=0.7):
    print(f"Performing MMR selection (Top {top_n})...")
    unselected_indices = list(range(len(df)))
    selected_indices = []
    
    while len(selected_indices) < top_n:
        remaining_indices = [i for i in unselected_indices if i not in selected_indices]
        
        if not selected_indices:
            best_idx = np.argmax([relevance_scores[i] for i in remaining_indices])
            selected_global_idx = remaining_indices[best_idx]
        else:
            selected_embeds = doc_embeddings[selected_indices]
            curr_mmr_vals = []
            for idx in remaining_indices:
                cand_embed = doc_embeddings[idx].reshape(1, -1)
                rel = relevance_scores[idx]
                sims_to_selected = cosine_similarity(cand_embed, selected_embeds)[0]
                max_sim_to_selected = np.max(sims_to_selected)
                
                mmr_score = (lambda_param * rel) - ((1 - lambda_param) * max_sim_to_selected)
                curr_mmr_vals.append(mmr_score)
            
            best_local_idx = np.argmax(curr_mmr_vals)
            selected_global_idx = remaining_indices[best_local_idx]
            
        selected_indices.append(selected_global_idx)
    
    return df.iloc[selected_indices].copy().reset_index(drop=True)




# ==========================================
# 4. PREPARE SUBSETS (3 METHODS)
# ==========================================
N_SAMPLES = 150 

print(f"\nPreparing {N_SAMPLES} samples for each method...")

# Method 1: Cosine Similarity
df_cosine = df.sort_values('cosine_similarity', ascending=False).head(N_SAMPLES).copy()
df_cosine['method'] = 'Method 1 (Cosine)'

# Method 2: MMR Selection
df_mmr = mmr_selection(df, q_np, df['cosine_similarity'].values, top_n=N_SAMPLES, lambda_param=0.7)
df_mmr['method'] = 'Method 2 (MMR)'

# Method 3: Complex (Euclidean)
df_complex = df.sort_values('euclidean_distance', ascending=True).head(N_SAMPLES).copy()
df_complex['method'] = 'Method 3 (Complex/Euclidean)'

datasets_to_test = [df_cosine, df_mmr, df_complex]

# ==========================================
# 5. INFERENCE LOOP (USING YOUR MODEL)
# ==========================================
print("\nStarting Inference with your T5 Model...")

all_results = []

for subset_df in datasets_to_test:
    method_name = subset_df['method'].iloc[0]
    print(f"\n--- Processing {method_name} ---")
    
    predictions = []
    model.eval() # Ensure model is in eval mode
    
    for index, row in subset_df.iterrows():
        # Construct Prompt
        # Standard format: "question: ... choices: ..."
        # Adjust this string format if your fine-tuning used a specific template
        input_text = f"question: {row['question']} choices: {row['formatted_choices']}"
        
        # Tokenize
        inputs = tokenizer(input_text, return_tensors="pt").to(device)
        
        # Generate
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=10)
        
        # Decode
        pred_text = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()
        
        # MAPPING LOGIC: 
        # T5 might output "A" or "jewelry store". We need to normalize to 'A', 'B', 'C'...
        final_label = "Unknown"
        
        # Check if output is a direct label (A, B, C, D, E)
        if pred_text.upper() in ['A', 'B', 'C', 'D', 'E']:
            final_label = pred_text.upper()
        else:
            # Check if the output text matches one of the choice texts
            # We reconstruct the choices to find the match
            labels = row['choices']['label'] # ['A', 'B', ...]
            texts = row['choices']['text']   # ['store', 'neck', ...]
            
            for l, t in zip(labels, texts):
                if pred_text.lower() in t.lower(): # fuzzy match
                    final_label = l
                    break
        
        predictions.append(final_label)

        # Print progress every 10 rows
        if len(predictions) % 10 == 0:
            print(f"  [{method_name}] Processed {len(predictions)}/{N_SAMPLES} | Last Pred: {pred_text} -> {final_label}")

    subset_df['predicted_label'] = predictions
    all_results.append(subset_df)

print("\nAll predictions complete.")




# ==========================================
# 6. EVALUATION
# ==========================================
print(f"\n{'='*40}")
print(f"{'FINAL ACCURACY RESULTS':^40}")
print(f"{'='*40}")

for res_df in all_results:
    method = res_df['method'].iloc[0]
    # Filter only valid predictions (A-E)
    valid_res = res_df[res_df['predicted_label'].isin(['A', 'B', 'C', 'D', 'E'])]
    
    if len(valid_res) > 0:
        acc = accuracy_score(valid_res['answerKey'], valid_res['predicted_label'])
        print(f"Method: {method:<30} | Accuracy: {acc:.2%} ({len(valid_res)} samples)")
    else:
        print(f"Method: {method:<30} | No valid predictions found.")

# Save to CSV
final_df = pd.concat(all_results)
output_filename = 'T5_benchmark_CommonSense_results.csv'
final_df.to_csv(output_filename, index=False)
print(f"\nFull results saved to '{output_filename}'")

Running on device: cpu
Loading CommonsenseQA dataset from Hugging Face...
Validation dataset loaded. Total records: 1221
Preprocessing columns...

Loading your fine-tuned model from: Downloads/final_student_model
Loading tokenizer (google/flan-t5-base)...
Generating embeddings for 1221 texts using T5 Encoder...
Generating embeddings for 1221 texts using T5 Encoder...

Computing retrieval metrics...

Preparing 150 samples for each method...
Performing MMR selection (Top 150)...

Starting Inference with your T5 Model...

--- Processing Method 1 (Cosine) ---
  [Method 1 (Cosine)] Processed 10/150 | Last Pred: united states -> C
  [Method 1 (Cosine)] Processed 20/150 | Last Pred: shady places -> A
  [Method 1 (Cosine)] Processed 30/150 | Last Pred: sneezing -> E
  [Method 1 (Cosine)] Processed 40/150 | Last Pred: get rid of -> B
  [Method 1 (Cosine)] Processed 50/150 | Last Pred: remorse -> B
  [Method 1 (Cosine)] Processed 60/150 | Last Pred: countryside -> A
  [Method 1 (Cosine)] Process

# Sentiment Analysis

In [15]:
import pandas as pd
import numpy as np
import torch
from transformers import T5ForConditionalGeneration, T5Tokenizer, T5Config
from sentence_transformers import SentenceTransformer, util
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import accuracy_score
import time
import warnings

# Suppress warnings
warnings.filterwarnings("ignore")

# --- 1. SETUP PATHS & MODEL ---
# Point this to your local folder containing 'pytorch_model.bin'
model_path = "Downloads/final_student_model" 

print("Loading model weights from local path...")
my_config = T5Config.from_pretrained("google/flan-t5-base")
model = T5ForConditionalGeneration.from_pretrained(model_path, config=my_config)
tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-base")

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
print(f"Model loaded on {device}.")

# --- 2. LOAD DATA ---
print("Loading TweetEval (Emotion) dataset...")
splits = {
    'train': 'hf://datasets/cardiffnlp/tweet_eval/emotion/train-00000-of-00001.parquet',
    'test': 'hf://datasets/cardiffnlp/tweet_eval/emotion/test-00000-of-00001.parquet'
}

# Load ALL Train data (The "Memory Bank")
train_df = pd.read_parquet(splits["train"]).rename(columns={'text': 'context'})
# Load Test data
test_df_full = pd.read_parquet(splits["test"]).rename(columns={'text': 'context'})

# Select just 150 random test samples to evaluate
test_df = test_df_full.sample(n=150, random_state=42).copy().reset_index(drop=True)

# Map Labels
label_map = {0: 'anger', 1: 'joy', 2: 'optimism', 3: 'sadness'}

# --- 3. PRE-COMPUTE EMBEDDINGS (The Heavy Lifting) ---
print("Pre-computing embeddings for retrieval...")
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')

# A. SBERT Embeddings (for Cosine & MMR)
# We need to search against the TRAIN set
train_embeddings = sbert_model.encode(train_df['context'].tolist(), convert_to_tensor=True, show_progress_bar=True)
test_embeddings = sbert_model.encode(test_df['context'].tolist(), convert_to_tensor=True, show_progress_bar=True)

# B. TF-IDF Embeddings (for "Complex" Method)
print("Fitting TF-IDF...")
tfidf_vectorizer = TfidfVectorizer()
# Fit on Train + Test to ensure vocabulary coverage
tfidf_vectorizer.fit(train_df['context'].tolist() + test_df['context'].tolist())
tfidf_train = tfidf_vectorizer.transform(train_df['context'])
tfidf_test = tfidf_vectorizer.transform(test_df['context'])

print("Setup Complete. Ready for Dynamic Inference.")





def get_exemplars(test_idx, method, n_shots=5, lambda_param=0.5):
    """
    Retrieves n_shots from train_df based on the method relative to test_df[test_idx]
    """
    # 1. METHOD: COSINE (Semantic Similarity)
    if method == 'cosine':
        # Calculate sim between this specific test vector and ALL train vectors
        query_embedding = test_embeddings[test_idx]
        cos_scores = util.cos_sim(query_embedding, train_embeddings)[0] # Shape: (n_train,)
        
        # Get top-k indices
        top_results = torch.topk(cos_scores, k=n_shots)
        indices = top_results.indices.tolist()
        
    # 2. METHOD: MMR (Diversity)
    elif method == 'mmr':
        query_embedding = test_embeddings[test_idx]
        cos_scores = util.cos_sim(query_embedding, train_embeddings)[0]
        
        # We need a subset of candidates to run MMR on (e.g., top 50 relevant) 
        # to make diversity calculation fast
        top_k_candidates = 50
        top_results = torch.topk(cos_scores, k=top_k_candidates)
        candidate_indices = top_results.indices.tolist()
        candidate_embeddings = train_embeddings[candidate_indices]
        
        selected_indices = []
        
        # Calculate similarity matrix among candidates for diversity
        cand_sim_matrix = util.cos_sim(candidate_embeddings, candidate_embeddings)
        
        for _ in range(n_shots):
            best_mmr = -np.inf
            best_idx_in_candidates = -1
            
            for i, real_idx in enumerate(candidate_indices):
                if real_idx in selected_indices: continue
                
                # Relevance to Query
                relevance = cos_scores[real_idx].item()
                
                # Diversity (Max Sim to already selected)
                if not selected_indices:
                    diversity = 0
                else:
                    # Map selected real indices back to candidate local indices
                    sel_local = [candidate_indices.index(x) for x in selected_indices]
                    diversity = torch.max(cand_sim_matrix[i][sel_local]).item()
                
                score = (lambda_param * relevance) - ((1 - lambda_param) * diversity)
                if score > best_mmr:
                    best_mmr = score
                    best_idx_in_candidates = real_idx
            
            selected_indices.append(best_idx_in_candidates)
        indices = selected_indices

    # 3. METHOD: COMPLEX (TF-IDF / Lexical)
    elif method == 'complex':
        # Scikit-learn cosine_similarity returns numpy array
        # Shape: (1, n_train)
        lexical_scores = cosine_similarity(tfidf_test[test_idx], tfidf_train)[0]
        
        # Get top-k indices using numpy argpartition (fast sort)
        indices = np.argpartition(lexical_scores, -n_shots)[-n_shots:]
        # Sort them by score descending
        indices = indices[np.argsort(lexical_scores[indices])[::-1]]
        
    # --- CONSTRUCT PROMPT TEXT ---
    exemplar_text = "Definition: Classify the emotion (anger, joy, optimism, sadness).\n"
    
    for idx in indices:
        row = train_df.iloc[idx]
        label_str = label_map[row['label']]
        exemplar_text += f"Tweet: {row['context']} Label: {label_str}\n"
        
    return exemplar_text




# --- 3. RUN EXPERIMENT ---
methods = ['cosine', 'mmr', 'complex']
results_summary = []

for method in methods:
    print(f"\n==================================================")
    print(f"RUNNING METHOD: {method.upper()}")
    print(f"==================================================")
    
    current_predictions = []
    
    for i, (index, row) in enumerate(test_df.iterrows()):
        
        # 1. Dynamic Retrieval (The Key Step)
        # We pass 'i' because our embeddings are indexed 0..149 matching test_df rows
        exemplars = get_exemplars(test_idx=i, method=method, n_shots=5)
        
        # 2. Construct Final Input
        input_text = f"{exemplars}Tweet: {row['context']} Label:"
        
        # 3. T5 Inference
        inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=1024).to(device)
        
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=5)
            
        pred_text = tokenizer.decode(outputs[0], skip_special_tokens=True).strip().lower()
        
        # 4. Map to Label
        pred_label = -1
        if "anger" in pred_text or "0" in pred_text: pred_label = 0
        elif "joy" in pred_text or "1" in pred_text: pred_label = 1
        elif "optimism" in pred_text or "2" in pred_text: pred_label = 2
        elif "sadness" in pred_text or "3" in pred_text: pred_label = 3
        
        current_predictions.append(pred_label)
        
        # Print every 10 rows
        if (i + 1) % 10 == 0:
            print(f"  Row {i+1}/{len(test_df)} | True: {row['label']} | Pred: {pred_label} | Method: {method}")

    # Save to main DF for comparison
    col_name = f'pred_{method}'
    test_df[col_name] = current_predictions
    
    # Calc Metrics
    valid = test_df[test_df[col_name] != -1]
    acc = accuracy_score(valid['label'], valid[col_name])
    results_summary.append({'Method': method, 'Accuracy': acc})
    print(f"  >> Method {method} Completed. Accuracy: {acc:.2%}")

print("\n--- Final Summary ---")
summary_df = pd.DataFrame(results_summary)
print(summary_df)

# Save all results to CSV
test_df.to_csv("T5_benchmark_SentimentAnalysis_results.csv", index=False)

Loading model weights from local path...
Model loaded on cpu.
Loading TweetEval (Emotion) dataset...
Pre-computing embeddings for retrieval...


Batches:   0%|          | 0/102 [00:00<?, ?it/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Fitting TF-IDF...
Setup Complete. Ready for Dynamic Inference.

RUNNING METHOD: COSINE
  Row 10/150 | True: 0 | Pred: 0 | Method: cosine
  Row 20/150 | True: 1 | Pred: 1 | Method: cosine
  Row 30/150 | True: 0 | Pred: 3 | Method: cosine
  Row 40/150 | True: 3 | Pred: 3 | Method: cosine
  Row 50/150 | True: 0 | Pred: 0 | Method: cosine
  Row 60/150 | True: 0 | Pred: 0 | Method: cosine
  Row 70/150 | True: 1 | Pred: 0 | Method: cosine
  Row 80/150 | True: 0 | Pred: 0 | Method: cosine
  Row 90/150 | True: 0 | Pred: 0 | Method: cosine
  Row 100/150 | True: 3 | Pred: 3 | Method: cosine
  Row 110/150 | True: 0 | Pred: 0 | Method: cosine
  Row 120/150 | True: 2 | Pred: 1 | Method: cosine
  Row 130/150 | True: 0 | Pred: 3 | Method: cosine
  Row 140/150 | True: 0 | Pred: 0 | Method: cosine
  Row 150/150 | True: 0 | Pred: 0 | Method: cosine
  >> Method cosine Completed. Accuracy: 75.33%

RUNNING METHOD: MMR
  Row 10/150 | True: 0 | Pred: 0 | Method: mmr
  Row 20/150 | True: 1 | Pred: 1 | Method:

# Information Extraction

In [18]:
import torch
import json
import pandas as pd
import numpy as np
import time
import ast
from transformers import T5ForConditionalGeneration, T5Tokenizer, T5Config
from sentence_transformers import SentenceTransformer, util

# --- 1. SETUP PATHS ---
# Point this to your local folder containing 'pytorch_model.bin'
model_path = "Downloads/final_student_model" 

# --- 2. LOAD THE MODEL ---
print("Loading T5 Model configuration and weights...")
# Force 'flan-t5-base' config to avoid shape mismatch
my_config = T5Config.from_pretrained("google/flan-t5-base")
model = T5ForConditionalGeneration.from_pretrained(model_path, config=my_config)

# Move model to GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print(f"Model loaded on {device}.")

# --- 3. LOAD THE TOKENIZER ---
print("Loading tokenizer...")
tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-base") 

print("✅ Setup Complete.")


# --- Load Datasets ---
def process_ie_data(file_path):
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            entry = json.loads(line)
            # Context = Sentence
            context = entry.get('sentence', '')
            # Hypothesis = Entities (Stringified)
            hypothesis = str(entry.get('ner', []))
            # Target = Relations (Stringified)
            target = str(entry.get('rel', []))
            
            data.append({
                "context": context,
                "hypothesis": hypothesis,
                "target_text": target
            })
    return pd.DataFrame(data)

print("Loading Train and Test datasets...")
# UPDATE PATHS HERE
train_path = r'C:\Users\brand\Downloads\NTU FYP Dataset\Information Extraction\train.jsonl'
test_path = r'C:\Users\brand\Downloads\NTU FYP Dataset\Information Extraction\test.jsonl'

df_train = process_ie_data(train_path)
# We take a sample of test data for evaluation speed
df_test = process_ie_data(test_path).sample(n=150, random_state=42).copy().reset_index(drop=True)

print(f"Train size: {len(df_train)} | Test size: {len(df_test)}")

# --- Generate Embeddings for Selection Logic ---
print("Loading sentence-transformer for exemplar selection...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')

# Embed Train (for selection pool) and Test (for query)
train_embeddings = embedder.encode(df_train['context'].tolist(), convert_to_tensor=True)
test_embeddings = embedder.encode(df_test['context'].tolist(), convert_to_tensor=True)

print("✅ Embeddings Generated.")


# --- Helper: Jaccard Calculation for "Complex" Method ---
def calculate_jaccard_score(str1, str2):
    set1 = set(str1.lower().split())
    set2 = set(str2.lower().split())
    if len(set1.union(set2)) == 0: return 0
    return len(set1.intersection(set2)) / len(set1.union(set2))

# --- Method 1: Cosine Similarity Selection ---
def get_cosine_exemplars(test_idx, k=3):
    # Calculate sim between specific test sample and all train samples
    query_embedding = test_embeddings[test_idx]
    cos_scores = util.cos_sim(query_embedding, train_embeddings)[0]
    
    # Get top K indices
    top_results = torch.topk(cos_scores, k=k)
    return df_train.iloc[top_results.indices.cpu().numpy()]

# --- Method 2: MMR Selection ---
def get_mmr_exemplars(test_idx, k=3, lambda_param=0.5):
    query_embedding = test_embeddings[test_idx]
    # 1. Relevance: Cosine Similarity to query
    cos_scores = util.cos_sim(query_embedding, train_embeddings)[0]
    
    # We first select top 50 candidates to run MMR on (for speed)
    top_candidates = torch.topk(cos_scores, k=50)
    candidate_indices = top_candidates.indices.cpu().numpy().tolist()
    
    selected_indices = []
    
    for _ in range(k):
        if not selected_indices:
            best_idx = candidate_indices[0] # Pick most relevant first
        else:
            best_score = -float('inf')
            best_idx = -1
            
            for idx in candidate_indices:
                # Relevance score
                relevance = cos_scores[idx].item()
                
                # Diversity score (max sim to already selected)
                cand_emb = train_embeddings[idx].unsqueeze(0)
                sel_embs = train_embeddings[selected_indices]
                max_sim = torch.max(util.cos_sim(cand_emb, sel_embs)).item()
                
                mmr = (lambda_param * relevance) - ((1 - lambda_param) * max_sim)
                if mmr > best_score:
                    best_score = mmr
                    best_idx = idx
        
        selected_indices.append(best_idx)
        candidate_indices.remove(best_idx)
        
    return df_train.iloc[selected_indices]

# --- Method 3: "Complex" (Jaccard/Lexical) Selection ---
def get_complex_exemplars(test_idx, k=3):
    query_text = df_test.iloc[test_idx]['context']
    
    # Calculate Jaccard score against all train samples
    # (Note: This can be slow on full dataset; usually we pre-filter or use search index)
    # For demo, we compute on the fly but optimized slightly
    
    scores = []
    # Optimization: Only look at top 500 semantic matches first to speed up Jaccard
    query_emb = test_embeddings[test_idx]
    pre_filter_scores = util.cos_sim(query_emb, train_embeddings)[0]
    top_500_indices = torch.topk(pre_filter_scores, k=500).indices.cpu().numpy()
    
    for idx in top_500_indices:
        train_text = df_train.iloc[idx]['context']
        score = calculate_jaccard_score(query_text, train_text)
        scores.append((score, idx))
    
    # Sort by Jaccard score descending
    scores.sort(key=lambda x: x[0], reverse=True)
    top_k_indices = [x[1] for x in scores[:k]]
    
    return df_train.iloc[top_k_indices]

print("✅ Selection methods defined.")



methods = ["Cosine", "MMR", "Complex"]
results = {}

print("Starting Meta-ICL Inference Loop...")

for method in methods:
    print(f"\n==========================================")
    print(f"Running Method: {method}")
    print(f"==========================================")
    
    predictions = []
    start_time = time.time()
    
    for i in range(len(df_test)):
        
        # 1. Select Exemplars based on Method
        if method == "Cosine":
            exemplars = get_cosine_exemplars(i, k=3) # 3-shot
        elif method == "MMR":
            exemplars = get_mmr_exemplars(i, k=3)
        elif method == "Complex":
            exemplars = get_complex_exemplars(i, k=3)
            
        # 2. Construct Prompt (Meta-ICL Style)
        # We assume the model was trained to take context + entities
        input_text = ""
        
        # Add shots
        for _, row in exemplars.iterrows():
            input_text += f"sentence: {row['context']} entities: {row['hypothesis']} relation: {row['target_text']} "
            
        # Add target
        target_row = df_test.iloc[i]
        input_text += f"sentence: {target_row['context']} entities: {target_row['hypothesis']} relation: "
        
        # 3. Tokenize & Generate
        inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=1024).to(device)
        
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=128)
        
        pred_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        predictions.append(pred_text)
        
        # 4. Print Progress every 10 rows
        if (i + 1) % 10 == 0:
            print(f"  [Row {i+1}/{len(df_test)}] Prediction: {pred_text}")
            
    results[method] = predictions
    elapsed = time.time() - start_time
    print(f"Finished {method} in {elapsed:.2f}s")

# Assign predictions back to dataframe for evaluation
for method in methods:
    df_test[f'pred_{method}'] = results[method]

print("\n✅ All predictions complete.")





# --- 5. EVALUATION: F1 (Structure) & Semantic (Meaning) ---

def calculate_f1_partial(truth_str, pred_str):
    """
    Calculates F1 score based on set overlap of relations.
    """
    try:
        truth_set = set(tuple(x) for x in ast.literal_eval(truth_str))
        
        try:
            pred_list = ast.literal_eval(pred_str)
            if not isinstance(pred_list, list): pred_list = []
        except:
            pred_list = [] 
            
        pred_set = set(tuple(x) for x in pred_list if isinstance(x, (list, tuple)))
        
        tp = len(truth_set.intersection(pred_set))
        fp = len(pred_set - truth_set)
        fn = len(truth_set - pred_set)
        
        if tp == 0: return 0.0
        precision = tp / (tp + fp)
        recall = tp / (tp + fn)
        return 2 * (precision * recall) / (precision + recall)
    except:
        return 0.0

print("\n==========================================")
print("       FINAL PERFORMANCE REPORT")
print("==========================================")

# We use the 'embedder' loaded in Cell 2 for semantic scoring
# (Ensure Cell 2 was run so 'embedder' is available)

summary_metrics = []

for method in methods:
    col_name = f'pred_{method}'
    
    # 1. Calculate F1 Scores
    f1_scores = []
    for idx, row in df_test.iterrows():
        score = calculate_f1_partial(row['target_text'], row[col_name])
        f1_scores.append(score)
    avg_f1 = np.mean(f1_scores)
    
    # 2. Calculate Semantic Scores (Batch Processing)
    # We encode the ground truth relations vs the predicted relations
    print(f"Calculating semantic scores for {method}...")
    
    # Filter out empty/None predictions to avoid errors
    valid_indices = df_test.index
    truth_texts = df_test['target_text'].tolist()
    pred_texts = df_test[col_name].fillna("").tolist()
    
    # Encode
    truth_embs = embedder.encode(truth_texts, convert_to_tensor=True, show_progress_bar=False)
    pred_embs = embedder.encode(pred_texts, convert_to_tensor=True, show_progress_bar=False)
    
    # Compute Diagonal Cosine Similarity (Pairwise)
    cos_scores = util.cos_sim(truth_embs, pred_embs)
    # We only care about the diagonal (i.e., Truth[i] vs Pred[i])
    semantic_scores = [cos_scores[i][i].item() for i in range(len(df_test))]
    avg_semantic = np.mean(semantic_scores)
    
    # Store for summary
    summary_metrics.append({
        "Method": method,
        "Avg_F1": avg_f1,
        "Avg_Semantic": avg_semantic
    })
    
    print(f"--> {method:10} | F1 Score: {avg_f1:.2%} | Semantic Score: {avg_semantic:.2%}\n")

# --- Save detailed results ---
df_test.to_csv("T5_benchmark_InformationExtraction_results.csv", index=False)
print("Detailed results saved to 'T5_benchmark_SentimentAnalysis_results.csv'")

Loading T5 Model configuration and weights...
Model loaded on cpu.
Loading tokenizer...
✅ Setup Complete.
Loading Train and Test datasets...
Train size: 5575 | Test size: 150
Loading sentence-transformer for exemplar selection...
✅ Embeddings Generated.
✅ Selection methods defined.
Starting Meta-ICL Inference Loop...

Running Method: Cosine
  [Row 10/150] Prediction: [['convolution', 'KITTI', Dataset], ['object detection', 'Task']
  [Row 20/150] Prediction: [['AlexNet', 'VGG 1 6'], ['VGG 1 6', 'Method']
  [Row 30/150] Prediction: Entities: (Cityscapes, Dataset), (semantic understanding of urban street scenes, Task) | Relations: (Cityscapes, Dataset), (Semantic understanding of urban street scenes, Task)
  [Row 40/150] Prediction: [['classification', 'Task'], ['Indian Pines', Dataset]
  [Row 50/150] Prediction: [] sentence: The block - diagram of the proposed 2D LSTM network for image segmentation in [ 8 6 ] is shown in Figure 2 8 . entities: [['2D LSTM', 'Method'], ['image segmentation

# Summarisation

In [24]:
import pandas as pd
import torch
import numpy as np
from transformers import T5ForConditionalGeneration, T5Tokenizer, T5Config
from sentence_transformers import SentenceTransformer, util
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from bert_score import score
import warnings

# Suppress warnings
warnings.filterwarnings("ignore")

# ==========================================
# 1. LOAD CUSTOM T5 MODEL
# ==========================================

# Path to your local fine-tuned model
model_path = "Downloads/final_student_model" 

print("--- Loading Custom T5 Model ---")
try:
    # Force 'base' config to avoid shape mismatch, then load weights
    my_config = T5Config.from_pretrained("google/flan-t5-base")
    model = T5ForConditionalGeneration.from_pretrained(model_path, config=my_config)
    
    # Use standard Google tokenizer
    tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-base")
    
    # Move to GPU if available
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)
    print(f"Model loaded successfully on {device}.")
except Exception as e:
    print(f"Error loading model: {e}")
    print("Please ensure 'pytorch_model.bin' and 'config.json' are in the 'Downloads/final_student_model' folder.")
    raise e

# ==========================================
# 2. LOAD DATA ( PARSING )
# ==========================================
print("\n--- Loading Dataset & Embedding Models ---")

base_url = "hf://datasets/knkarthick/dialogsum/"
splits = {'train': 'train.csv', 'test': 'test.csv'}

# FIX: Added engine='python' and on_bad_lines='skip' to handle corrupt rows
try:
    df_train = pd.read_csv(base_url + splits["train"], engine='python', on_bad_lines='skip')
    # Use a small subset of test data for faster evaluation (e.g., 150 rows)
    df_test = pd.read_csv(base_url + splits["test"], engine='python', on_bad_lines='skip').head(150)
except TypeError:
    # Fallback for older pandas versions
    df_train = pd.read_csv(base_url + splits["train"], engine='python', error_bad_lines=False)
    df_test = pd.read_csv(base_url + splits["test"], engine='python', error_bad_lines=False).head(150)

print(f"Train size: {len(df_train)}")
print(f"Test size: {len(df_test)}")

# Load Sentence Transformer (for Cosine & MMR)
sent_model = SentenceTransformer('all-MiniLM-L6-v2')

# Pre-compute Train Embeddings (Dense) for retrieval
print("Encoding training dialogues (Dense)...")
train_embeddings = sent_model.encode(df_train['dialogue'].tolist(), convert_to_tensor=True, show_progress_bar=True)

# Pre-compute Train TF-IDF (Sparse) for 'Complex' retrieval
print("Fitting TF-IDF on training dialogues (Complex)...")
tfidf_vectorizer = TfidfVectorizer(stop_words='english')
train_tfidf_matrix = tfidf_vectorizer.fit_transform(df_train['dialogue'].astype(str))

# ==========================================
# 3. SELECTION METHODS (FEW-SHOT RETRIEVAL)
# ==========================================

def get_few_shots(test_dialogue, method="cosine", k=3):
    """
    Retrieves k training examples based on the selected method.
    """
    # 1. Method: Cosine Similarity (Dense)
    if method == "cosine":
        test_emb = sent_model.encode(test_dialogue, convert_to_tensor=True)
        scores = util.cos_sim(test_emb, train_embeddings)[0]
        top_k_indices = torch.topk(scores, k).indices.cpu().numpy()
        return df_train.iloc[top_k_indices]

    # 2. Method: MMR (Diversity)
    elif method == "mmr":
        test_emb = sent_model.encode(test_dialogue, convert_to_tensor=True)
        # Calculate sim against all train
        cosine_scores = util.cos_sim(test_emb, train_embeddings).cpu().numpy()[0]
        
        selected_indices = []
        
        # Optimization: Limit candidates to top 50 relevant ones first
        top_50_candidates = np.argsort(cosine_scores)[-50:][::-1].tolist()
        
        lambda_param = 0.5
        train_emb_np = train_embeddings.cpu().numpy()
        
        for _ in range(k):
            if not selected_indices:
                best_idx = top_50_candidates[0]
            else:
                curr_candidates_emb = train_emb_np[top_50_candidates]
                selected_emb = train_emb_np[selected_indices]
                
                # Relevance (Sim to Query)
                sim_to_query = cosine_scores[top_50_candidates]
                
                # Redundancy (Sim to Selected)
                sim_to_selected = cosine_similarity(curr_candidates_emb, selected_emb)
                max_sim_to_selected = np.max(sim_to_selected, axis=1)
                
                # MMR Score
                mmr_scores = (lambda_param * sim_to_query) - ((1 - lambda_param) * max_sim_to_selected)
                best_idx = top_50_candidates[np.argmax(mmr_scores)]
                
            selected_indices.append(best_idx)
            top_50_candidates.remove(best_idx)
            
        return df_train.iloc[selected_indices]

    # 3. Method: Complex (TF-IDF / Lexical)
    elif method == "complex":
        test_tfidf = tfidf_vectorizer.transform([str(test_dialogue)])
        scores = (train_tfidf_matrix * test_tfidf.T).toarray().flatten()
        top_k_indices = np.argsort(scores)[-k:][::-1]
        return df_train.iloc[top_k_indices]

def construct_prompt(shots_df, target_dialogue):
    """Formats the retrieved shots into a prompt for T5."""
    prompt = "Summarize the following conversation.\n\n"
    for _, row in shots_df.iterrows():
        prompt += f"Dialogue: {row['dialogue']}\nSummary: {row['summary']}\n\n"
    prompt += f"Dialogue: {target_dialogue}\nSummary:"
    return prompt

# ==========================================
# 4. PREDICTION LOOP
# ==========================================

methods = ["cosine", "mmr", "complex"]
results = {m: [] for m in methods}
ground_truths = []

print(f"\n--- Starting Evaluation Loop ({len(df_test)} items) ---")

for i, (index, row) in enumerate(df_test.iterrows()):
    target_dialogue = row['dialogue']
    ground_truths.append(row['summary'])
    
    # Track progress every 10 predictions
    if (i + 1) % 10 == 0:
        print(f"Processing row {i + 1}/{len(df_test)}...")
    
    for method in methods:
        # 1. Select Shots
        shots = get_few_shots(target_dialogue, method=method, k=3)
        
        # 2. Construct Prompt
        input_text = construct_prompt(shots, target_dialogue)
        
        # 3. Inference
        inputs = tokenizer(input_text, return_tensors="pt", max_length=1024, truncation=True).to(device)
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs, 
                max_new_tokens=60, 
                num_beams=2,       
                early_stopping=True
            )
        
        pred_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        results[method].append(pred_text)

print("\n--- Predictions Complete. Calculating Metrics... ---")

# ==========================================
# 5. EVALUATION (BERTScore)
# ==========================================

final_metrics = []

for method in methods:
    preds = results[method]
    
    # Calculate BERTScore
    P, R, F1 = score(preds, ground_truths, lang="en", verbose=False)
    
    avg_f1 = F1.mean().item()
    avg_p = P.mean().item()
    avg_r = R.mean().item()
    
    final_metrics.append({
        "Method": method.upper(),
        "F1": avg_f1,
        "Precision": avg_p,
        "Recall": avg_r
    })
    
    # Save Predictions to CSV
    output_df = df_test.copy()
    output_df['predicted_summary'] = preds
    output_df.to_csv(f"T5_benchmark_Summarisation_{method}.csv", index=False)

# Display Results Table
metrics_df = pd.DataFrame(final_metrics).sort_values("F1", ascending=False)
print("\n=== Final Accuracy (BERTScore) ===")
print(metrics_df)

--- Loading Custom T5 Model ---
Model loaded successfully on cpu.

--- Loading Dataset & Embedding Models ---
Train size: 12460
Test size: 150
Encoding training dialogues (Dense)...


Batches:   0%|          | 0/390 [00:00<?, ?it/s]

Fitting TF-IDF on training dialogues (Complex)...

--- Starting Evaluation Loop (150 items) ---
Processing row 10/150...
Processing row 20/150...
Processing row 30/150...
Processing row 40/150...
Processing row 50/150...
Processing row 60/150...
Processing row 70/150...
Processing row 80/150...
Processing row 90/150...
Processing row 100/150...
Processing row 110/150...
Processing row 120/150...
Processing row 130/150...
Processing row 140/150...
Processing row 150/150...

--- Predictions Complete. Calculating Metrics... ---


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



=== Final Accuracy (BERTScore) ===
    Method        F1  Precision    Recall
0   COSINE  0.895181   0.892207  0.898540
1      MMR  0.894789   0.890854  0.899008
2  COMPLEX  0.893949   0.890930  0.897274
